# GRPO on NFL spread picks
Runtime > Change runtime type > **T4 GPU**, then run top to bottom. Change `REWARD` to `"brier"` for the second experiment.

In [ ]:
REPO_URL = "https://github.com/ZekeRomero/nfl grpo.git"   # <- set after you push the repo
MODEL    = "unsloth/Qwen2.5-1.5B-Instruct"
REWARD   = "win"        # "win" or "brier"
MAX_STEPS = 300          # each step = 1 game x NUM_GEN sampled answers
NUM_GEN   = 8            # group size; drop to 4 if you run out of memory
MAX_NEW_TOKENS = 300
EVAL_GAMES = None        # e.g. 100 for a quick smoke test, None = all
SEED = 0

In [ ]:
!pip install -q unsloth
!git clone -q $REPO_URL nfl-grpo
%cd nfl-grpo
!python scripts/build_dataset.py
!python scripts/run_baselines.py

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
import json, torch, pandas as pd
from datasets import Dataset
from nflgrpo.data import split
from nflgrpo.prompts import VARIANTS, messages
from nflgrpo.rewards import REWARD_SETS, parse_output
from nflgrpo.evaluate import score, consistency, table

model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, random_state=SEED,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth")
tokenizer.padding_side = "left"

df = pd.read_csv("data/games_features.csv")
train_df, val_df, test_df = split(df)
train_ds = Dataset.from_list([json.loads(l) for l in open("data/train.jsonl")]).shuffle(seed=SEED)
print(len(train_ds), "training games")

## Evaluation helpers
Greedy decoding, so a given model always gives the same pick for the same prompt.

In [ ]:
@torch.no_grad()
def generate(msg_lists, batch_size=16):
    outs = []
    for i in range(0, len(msg_lists), batch_size):
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                 for m in msg_lists[i:i+batch_size]]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        outs += tokenizer.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return outs

def evaluate(part, tag, **prompt_kw):
    part = part.head(EVAL_GAMES) if EVAL_GAMES else part
    texts = generate([messages(r, **prompt_kw) for _, r in part.iterrows()])
    parsed = [parse_output(t) for t in texts]
    pd.DataFrame({"game_id": part.game_id.values, "label": part.label.values, "output": texts,
                  "pick": [p[0] for p in parsed], "conf": [p[1] for p in parsed]}
                 ).to_json(f"data/outputs_{tag}.jsonl", orient="records", lines=True)
    return score(part, [p[0] for p in parsed], [p[1] for p in parsed]), [p[0] for p in parsed]

def metamorphic(part, n=150):
    """Same game, reworded prompt: the pick should not change."""
    part = part.sample(min(n, len(part)), random_state=SEED)
    picks = {k: [parse_output(t)[0] for t in generate([messages(r, **kw) for _, r in part.iterrows()])]
             for k, kw in VARIANTS.items()}
    return consistency(picks)

## Before training

In [ ]:
FastLanguageModel.for_inference(model)
results = {}
results["untrained_val"], _ = evaluate(val_df, "untrained_val")
results["untrained_test"], _ = evaluate(test_df, "untrained_test")
meta_before = metamorphic(test_df)
display(table(results).round(2)); print(meta_before)

## GRPO training
Watch `reward` and the per-function rewards in the log. The format reward should climb fast; the outcome reward will be very noisy by design.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
FastLanguageModel.for_training(model)
args = GRPOConfig(
    output_dir=f"outputs_{REWARD}", seed=SEED, report_to="none",
    learning_rate=5e-6, lr_scheduler_type="cosine", warmup_ratio=0.1, weight_decay=0.01,
    num_generations=NUM_GEN, per_device_train_batch_size=NUM_GEN, gradient_accumulation_steps=1,
    max_completion_length=MAX_NEW_TOKENS, temperature=1.0,
    max_steps=MAX_STEPS, logging_steps=5, save_steps=100,
)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=args,
                      reward_funcs=REWARD_SETS[REWARD], train_dataset=train_ds)
trainer.train()
pd.DataFrame(trainer.state.log_history).to_csv(f"data/train_log_{REWARD}.csv", index=False)

## After training

In [ ]:
FastLanguageModel.for_inference(model)
results[f"grpo_{REWARD}_val"], _ = evaluate(val_df, f"grpo_{REWARD}_val")
results[f"grpo_{REWARD}_test"], _ = evaluate(test_df, f"grpo_{REWARD}_test")
meta_after = metamorphic(test_df)
t = table(results).round(2); t.to_csv(f"data/results_{REWARD}.csv"); display(t)
print("consistency before:", meta_before); print("consistency after: ", meta_after)
model.save_pretrained(f"lora_{REWARD}"); tokenizer.save_pretrained(f"lora_{REWARD}")